# KoBERT Fine-tuning 실습: 한국어 감정 인식

이 노트북은 Google Colab에서 KoBERT 계열 모델을 fine-tuning하여 한국어 감성분석/감정 인식을 수행하는 예제입니다.

- Task: 한국어 문장 긍정/부정 분류
- Dataset: NSMC 네이버 영화 리뷰 데이터셋
- Model: `skt/kobert-base-v1`
- Framework: Hugging Face Transformers + Datasets + Trainer

Colab 상단 메뉴에서 **런타임 > 런타임 유형 변경 > GPU**를 선택한 뒤 실행하세요.

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas matplotlib seaborn

## 1. 라이브러리 로드 및 환경 설정

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

# AUTO-INJECTED: Korean font setup for matplotlib
import os as _os
import matplotlib.font_manager as _fm
import matplotlib.pyplot as _plt
if not any('NanumGothic' in f.name for f in _fm.fontManager.ttflist):
    for _font in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
                  '/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf']:
        if _os.path.exists(_font):
            _fm.fontManager.addfont(_font)
_plt.rcParams.update({'font.family': 'NanumGothic', 'axes.unicode_minus': False})
del _os, _fm, _plt
# END AUTO-INJECTED Korean font setup


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

## 2. 데이터셋 로드: NSMC

In [ ]:
dataset = load_dataset("nsmc")
dataset

In [ ]:
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

print(train_df.head())
print(train_df["label"].value_counts())

In [ ]:
label_map = {0: "부정", 1: "긍정"}

for i in range(5):
    print("문장:", train_df.loc[i, "document"])
    print("라벨:", label_map[train_df.loc[i, "label"]])
    print("-" * 50)

## 3. 실습용 데이터 크기 축소

Colab 실습 시간을 줄이기 위해 subset을 사용합니다. 전체 데이터로 학습하려면 `select(range(...))` 부분을 제거하거나 범위를 늘리세요.

In [ ]:
small_train = dataset["train"].shuffle(seed=42).select(range(10000))
small_valid = dataset["test"].shuffle(seed=42).select(range(2000))
small_test = dataset["test"].shuffle(seed=123).select(range(2000))

print(small_train)
print(small_valid)
print(small_test)

## 4. KoBERT 모델 및 토크나이저 로드

In [ ]:
model_name = "skt/kobert-base-v1"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
    trust_remote_code=True
)

model.to(device)

### 대체 모델 옵션

KoBERT tokenizer 로드 문제가 발생하면 아래 셀의 주석을 해제하여 `klue/bert-base`를 사용할 수 있습니다.

In [ ]:
# model_name = "klue/bert-base"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
# model.to(device)

## 5. 전처리 및 토큰화

In [ ]:
def preprocess_function(examples):
    texts = []
    labels = []
    for text, label in zip(examples["document"], examples["label"]):
        if text is not None and len(text.strip()) > 0:
            texts.append(text)
            labels.append(label)
    tokenized = tokenizer(texts, truncation=True, max_length=128)
    tokenized["labels"] = labels
    return tokenized

tokenized_train = small_train.map(preprocess_function, batched=True, remove_columns=small_train.column_names)
tokenized_valid = small_valid.map(preprocess_function, batched=True, remove_columns=small_valid.column_names)
tokenized_test = small_test.map(preprocess_function, batched=True, remove_columns=small_test.column_names)

print(tokenized_train[0])

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 6. 평가 지표 정의

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

## 7. 학습 설정 및 Trainer 구성

In [ ]:
training_args = TrainingArguments(
    output_dir="./kobert-nsmc-sentiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

## 8. Fine-tuning 실행

In [ ]:
trainer.train()

## 9. 검증 및 테스트 평가

In [ ]:
valid_result = trainer.evaluate(tokenized_valid)
valid_result

In [ ]:
test_result = trainer.evaluate(tokenized_test)
test_result

## 10. 상세 성능 분석

In [ ]:
pred_output = trainer.predict(tokenized_test)
logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = np.argmax(logits, axis=-1)

print(classification_report(y_true, y_pred, target_names=["부정", "긍정"], digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["부정 예측", "긍정 예측"],
    yticklabels=["부정 실제", "긍정 실제"]
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("KoBERT Sentiment Classification Confusion Matrix")
plt.show()

## 11. 모델 저장 및 재로드

In [ ]:
save_dir = "./kobert-nsmc-finetuned"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved to:", save_dir)

In [ ]:
loaded_tokenizer = AutoTokenizer.from_pretrained(save_dir, trust_remote_code=True)
loaded_model = AutoModelForSequenceClassification.from_pretrained(save_dir, trust_remote_code=True)
loaded_model.to(device)
loaded_model.eval()

## 12. 단일 문장 감정 예측

In [ ]:
def predict_sentiment(text):
    inputs = loaded_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = loaded_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()
    return {
        "text": text,
        "prediction": label_map[pred],
        "negative_prob": probs[0][0].item(),
        "positive_prob": probs[0][1].item()
    }

examples = [
    "정말 감동적인 영화였습니다. 다시 보고 싶어요.",
    "시간이 너무 아까웠고 배우들의 연기도 별로였습니다.",
    "스토리는 평범했지만 음악과 영상미가 좋았습니다.",
    "최악입니다. 돈이 아까워요.",
    "기대 이상으로 재미있었습니다."
]

for text in examples:
    result = predict_sentiment(text)
    print(f"입력 문장: {result['text']}")
    print(f"예측 감정: {result['prediction']}")
    print(f"부정 확률: {result['negative_prob']:.4f}")
    print(f"긍정 확률: {result['positive_prob']:.4f}")
    print("-" * 60)

In [ ]:
user_text = input("감정을 분석할 한국어 문장을 입력하세요: ")
result = predict_sentiment(user_text)
print("입력 문장:", result["text"])
print("예측 감정:", result["prediction"])
print("부정 확률:", round(result["negative_prob"], 4))
print("긍정 확률:", round(result["positive_prob"], 4))

## 13. 오분류 샘플 분석

In [ ]:
test_texts = []
for item in small_test:
    text = item["document"]
    label = item["label"]
    if text is not None and len(text.strip()) > 0:
        test_texts.append((text, label))

error_cases = []
for i, (text, true_label) in enumerate(test_texts[:len(y_pred)]):
    pred_label = y_pred[i]
    if true_label != pred_label:
        error_cases.append({
            "text": text,
            "true_label": label_map[true_label],
            "pred_label": label_map[pred_label]
        })

error_df = pd.DataFrame(error_cases)
print("오분류 개수:", len(error_df))
print("오분류 비율:", len(error_df) / len(y_pred))
error_df.head(20)